<a href="https://colab.research.google.com/github/edavidbustosp/AplicacionesDeepLearning/blob/main/DeteccionFraude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Detección de Fraude Mediante Deep Learning**
Proyecto desarrollado por:

Ing. Edgar David Bustos Pulido

Aplicaciones de Deep Learning



#**Primer Avance**

Implementación inicial del modelo,

* Pruebas experimentales

* Análisis de resultados

* Propuesta de rediseño.


Importación de Librerías

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)


Configuraciones

In [ ]:
#Configuración de visualización
sns.set_theme(style="whitegrid")

#Configuración de reproducibilidad

SEED=42

np.random.seed(SEED)
torch.manual_seed(SEED)

#Dispositivo de ejecución
device=torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo de ejecución: ", device)
print("Versión de Pythorch: ", torch.__version__)


Carga del dataset

In [ ]:
from google.colab import files
from io import BytesIO

uploaded=files.upload()

archivo=next(iter(uploaded))

df=pd.read_csv(BytesIO(uploaded[archivo]))

print("Dataset cargado correctamente")
print("Dimensiones: ",df.shape)

display(df.head())

#**Analisis Exploratorio Inicial**

In [ ]:
# ======================================
# ANÁLISIS EXPLORATORIO DEL DATASET
# ======================================

#1. Información general

print("Información general del dataset")
print("-" * 50)

df.info()


#2. Verificación de valores nulos

print("\nValores Nulos")
print("-" * 50)

print("Total de valores nulos: ", df.isnull().sum().sum())

#3. Verificación de registros duplicados

print("\nRegistros duplicados")
print("-" * 50)

print("Cantidad de duplicados: ", df.duplicated().sum())

#4. Distribución de las clases
print("\nDistribución de las clases")
print("-" * 50)

conteo_clases=df["Class"].value_counts.sort_index()
porcentaje_clases=df["Class"].value_counts(normalize=True).sort_index()*100

distribucion=pd.DataFrame({
    "Cantidad":conteo_clases,
    "Porcentaje":porcentaje_clases
})

distribucion.idex=["Legitimas (0)", "Fraudulentas (1)"]

display(distribucion)

#5. Validación del desequilibrio
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="Class"
)

plt.title("Distribución de Transacciones legitimas y Fraudulentas")
plt.xlabel("Clase (0 = Legitima, 1 = Fraude)")
plt.ylabel("Cantidad de transacciones")

plt.show()


#6. Estadísticas descriptivas
print("\nEstadísticas descriptivas")
print("-" * 50)

display(df[["Time", "Amount"].describe])

